In [ ]:
#importation des modules
import torch
from torchvision.models import ResNet18_Weights
import torchvision.models as models
import numpy as np
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch.utils.data import random_split    
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import Subset
from torch.utils.data import WeightedRandomSampler
from collections import Counter
import os
import torch.nn as nn
import gc
from sklearn.metrics import roc_curve, auc


In [ ]:
#importation du dataset vindr et miniddsm
chemin_data_vindr="/home/onyxia/work/dataset_vindr" #à définir
chemin_data_miniddsm="/home/onyxia/data/MINI-DDSM/nouveau_dataset_mini_ddsm_700_700"

os.makedirs(chemin_data_vindr,exist_ok=True)
os.makedirs(os.path.join(chemin_data_vindr,"0-normal"),exist_ok=True)
os.makedirs(os.path.join(chemin_data_vindr,"1-cancer_benign"),exist_ok=True)

!mc mirror s3/lucasvital/stat_app/0-normal/ /home/onyxia/work/dataset_vindr/0-normal
!mc mirror s3/lucasvital/stat_app/1-cancer_benign/ /home/onyxia/work/dataset_vindr/1-cancer_benign


!mc mirror s3/lucasvital/stat_app/nouveau_dataset_mini_ddsm_700_700/ /home/onyxia/data/MINI-DDSM/nouveau_dataset_mini_ddsm_700_700

In [ ]:
#choix d'importation du modèle
!mc cp s3/lucasvital/stat_app/new2_best_model_672_general_standard_final_loss.pth /home/onyxia/work/best_resnet18_model.pth

In [ ]:
#choix des paramètres de génération des données (à choisir identique au modèle entraîné)
resolution=672 #resolution des images transformées
bs = 32 #batch size 

#choix de normalisation des images
choix_normalisation_image = 2 #à choisir entre 0- pas de normalisation, 1-normalisation_par_image 2-normalisation_sur_le_dataset 
limite_basse=0.05 #à choisir entre proche de 0 (on exclut le fond noir de la normalisation et on le fixe arbitrairement bas ensuite)


In [ ]:
# Génération des données équilibrées (standardisation robuste par image)

def generation_donnes_eval(chemin_data,mode): #mode = 1 si on veut évaluer le dataset, mode = 0 si on veut juste calculer les quantiles et la distribution 

    #définition de la fonction de pour normaliser les images par rapport à elles mêmes
    def robust_standardize1(x):
        mask = x > limite_basse         #On crée un masque pour ignorer le fond (souvent proche de 0 ou < 0.05)
        if mask.any():
            mean = x[mask].mean()
            std = x[mask].std()
            x[mask] = (x[mask] - mean) / (std + 1e-6) 
            x[mask] = torch.clamp(x[mask], -3, 3) #on ramène les valeurs extrêmes du sein à l'intervalle -3 3
        x[~mask] = -4.0 # on fixe le reste des valeurs correspondantes au support noir à -4
        return x


    #definition de la fonction pour normaliser les images par rapport au dataset MiniDDSM et calcul de mean et std error du dataset
    def calcul_dataset_mean_std():

        def recuperation_pixel(loader_source, n_batches=20):
            # On récupère quelques données pour avoir une distribution stable
            source_pixels = []
            
            with torch.no_grad():
                # Extraction du domaine Source
                for i, (images, _) in tqdm(enumerate(loader_source)):
                    if i >= n_batches: break
                    # On prend un seul canal (gris) et on aplatit
                    source_pixels.extend(images[:, 0, :, :].flatten().numpy())
            return(source_pixels)

        def robust_mean_variance(x):
            # 1. On calcule la std error et la mean sur le dataset
            mask = (x > limite_basse)  
            if mask.any():
                mean = x[mask].mean()
                std = x[mask].std()
            return (mean,std)

        train_transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor()])

        train_dataset_full = ImageFolder(root=chemin_data, transform=train_transform)
        train_loader1 = DataLoader(train_dataset_full, batch_size=bs)

        (source_pixel)=recuperation_pixel(train_loader1)
        source_pixel=np.array(source_pixel)

        (mu,sigma)=robust_mean_variance(source_pixel)
        return(mu,sigma)

    def robust_standardize2(x):
        mask = x > limite_basse
        if mask.any():
            mean = mu
            std = sigma
            x[mask] = (x[mask] - mean) / (std + 1e-6)    
            x[mask] = torch.clamp(x[mask], -3, 3)
        x[~mask] = -4.0 
        return x

    if choix_normalisation_image==0:
        def identity(x):
            return x
        fonction = identity
    elif choix_normalisation_image==1:
        fonction=robust_standardize1
    else:
        (mu,sigma)=calcul_dataset_mean_std()
        fonction=robust_standardize2
    
    if mode == 0:
        transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor(),
        transforms.Lambda(fonction)])

    elif mode == 1:
        transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor(),
        transforms.Lambda(fonction), 
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    data_set=ImageFolder(root=chemin_data, transform=transform)

    label=np.array(data_set.targets)
    print(f"nombre d'images : {len(label)}")
    print(f"proportion d'image benign-cancer : {label.sum()/len(label)}")

    test_loader = DataLoader(
        data_set, 
        batch_size=bs,       # Augmente le batch_size si possible (ex: 64 ou 128)
        shuffle=False, 
        num_workers=4,       # <--- ESSENTIEL : utilise plusieurs cœurs CPU
        pin_memory=True      # <--- Accélère le transfert CPU vers GPU
        )
    return(test_loader)

In [ ]:
#extraction des résultats sur test set ou sur val set

#récupération du dataset Vindr transformé pour l'évaluer
test_loader_vindr=generation_donnes_eval(chemin_data=chemin_data_vindr,mode=1)

#récupération du modèle
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
num_class=2
model.fc = nn.Sequential(
    nn.Dropout(p=0.5), # Désactive 50% des neurones aléatoirement à chaque itération
    nn.Linear(model.fc.in_features, num_class))

model.load_state_dict(torch.load("/home/onyxia/work/best_resnet18_model.pth"))

device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"GPU détecté : {torch.cuda.get_device_name(0)}")

def evaluation(choose_data):

    model.eval()

    all_predictions = []
    all_labels= []
    all_probabilities = []

    with torch.no_grad():
        for inputs, labels in tqdm(choose_data):
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            probabilities = F.softmax(outputs, dim=1)
            _,preds=torch.max(outputs,1)
            all_predictions.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            #pour le calcul de l'AUC, on peut aussi stocker les probabilités
            positive_class_probs = probabilities[:, 1].cpu().numpy()
            all_probabilities.extend(positive_class_probs)
            # Traitez les probabilités comme nécessaire

    predictions_np=np.array(all_predictions)
    labels_np=np.array(all_labels)
    probabilities_np=np.array(all_probabilities)


    accuracy = np.mean(predictions_np == labels_np)
    print(f'Test_Accuracy : {accuracy*100:.4f}%')

    #nettoyage VRAM
    gc.collect()
    torch.cuda.empty_cache()

    #matrice de confusion et AUC

    class_names = ['0-normal', '1-cancer_benign']
    cm = confusion_matrix(labels_np, predictions_np)

    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')   
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    plt.show()

    #calcul de l'AUC et de la courbe ROC
    fpr, tpr, thresholds = roc_curve(labels_np, probabilities_np, pos_label=1)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(4,4))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')

    plt.xlabel('False Positive Rate -Spécificity')
    plt.ylabel('True Positive Rate - Sensitivity')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

    return("évaluation terminée")


In [ ]:
evaluation(choose_data=test_loader_vindr)

In [ ]:
# Génération des data vindr matching quantile

def generation_donnes_eval_matching_quantile(seuil_pixel_fond_noir=-2.8,seuil_brillance=2.9): #mode = 1 si on veut évaluer le dataset, mode = 0 si on veut juste calculer les quantiles et la distribution 

    #définition de la fonction de pour normaliser les images par rapport à elles mêmes
    def robust_standardize1(x):
        mask = x > limite_basse         #On crée un masque pour ignorer le fond (souvent proche de 0 ou < 0.05)
        if mask.any():
            mean = x[mask].mean()
            std = x[mask].std()
            x[mask] = (x[mask] - mean) / (std + 1e-6) 
            x[mask] = torch.clamp(x[mask], -3, 3) #on ramène les valeurs extrêmes du sein à l'intervalle -3 3
        x[~mask] = -4.0 # on fixe le reste des valeurs correspondantes au support noir à -4
        return x


    #definition de la fonction pour normaliser les images par rapport au dataset MiniDDSM et calcul de mean et std error du dataset
    def calcul_dataset_mean_std():

        def recuperation_pixel(loader_source, n_batches=20):
            # On récupère quelques données pour avoir une distribution stable
            source_pixels = []
            
            with torch.no_grad():
                # Extraction du domaine Source
                for i, (images, _) in tqdm(enumerate(loader_source)):
                    if i >= n_batches: break
                    # On prend un seul canal (gris) et on aplatit
                    source_pixels.extend(images[:, 0, :, :].flatten().numpy())
            return(source_pixels)

        def robust_mean_variance(x):
            # 1. On calcule la std error et la mean sur le dataset
            mask = (x > limite_basse)  
            if mask.any():
                mean = x[mask].mean()
                std = x[mask].std()
            return (mean,std)

        train_transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor()])

        train_dataset_full = ImageFolder(root=chemin_data_vindr, transform=train_transform)
        train_loader1 = DataLoader(train_dataset_full, batch_size=bs)

        (source_pixel)=recuperation_pixel(train_loader1)
        source_pixel=np.array(source_pixel)

        (mu,sigma)=robust_mean_variance(source_pixel)
        return(mu,sigma)

    def robust_standardize2(x):
        mask = x > limite_basse
        if mask.any():
            mean = mu
            std = sigma
            x[mask] = (x[mask] - mean) / (std + 1e-6)    
            x[mask] = torch.clamp(x[mask], -3, 3)
        x[~mask] = -4.0 
        return x


    #------------------------------------------------------------------------------------------------------------------------
    #matching quantile : calcul des quantiles et représentations

    def calcul_quantile_et_representation(representation_graphique=True):
        def plot_distributions(loader_source, loader_target, n_batches=20):
            # On récupère quelques données pour avoir une distribution stable
            source_pixels = []
            target_pixels = []
            
            with torch.no_grad():
                # Extraction du domaine Source
                for i, (images, _) in tqdm(enumerate(loader_source)):
                    if i >= n_batches: break
                    # On prend un seul canal (gris) et on aplatit
                    source_pixels.extend(images[:, 0, :, :].flatten().numpy())
                    
                # Extraction du domaine Cible (VinDr)
                for i, (images, _) in tqdm(enumerate(loader_target)):
                    if i >= n_batches: break
                    target_pixels.extend(images[:, 0, :, :].flatten().numpy())
            
            return(source_pixels,target_pixels)


        #récupération des datasets Vindr et miniDDSM transformé 
        test_loader_vindr0=generation_donnes_eval(chemin_data=chemin_data_vindr,mode=0)
        test_loader_miniddsm0=generation_donnes_eval(chemin_data=chemin_data_miniddsm,mode=0)

        (source_pixel,target_pixel)=plot_distributions(test_loader_miniddsm0, test_loader_vindr0)

        if representation_graphique:
            #représentation graphique de la distribution (on raccourci les listes en divisant par 3 pour que la représentations graphiques ne prenne pas trop de temps)
            plt.figure(figsize=(10, 6))
            plt.hist(source_pixel[:int(len(source_pixel)/4)], bins=100, alpha=0.5, label='Source (Original)', color='blue', density=True)
            plt.hist(target_pixel[:int(len(source_pixel)/4)], bins=100, alpha=0.5, label='Target (VinDr)', color='red', density=True)

            plt.title("Comparaison des distributions d'intensité (Pixels)")
            plt.xlabel("Valeur du pixel (après normalisation)")
            plt.ylabel("Densité")
            plt.legend()
            plt.grid(alpha=0.3)
            plt.show()
        
        #ATTENTION : valeurs à ajuster
        source_pixel=np.array(source_pixel)
        target_pixel=np.array(target_pixel)
        source_pixel_breast=source_pixel[source_pixel> seuil_pixel_fond_noir]
        source_pixel_breast=source_pixel_breast[source_pixel_breast<seuil_brillance]
        target_pixel_breast=target_pixel[target_pixel> seuil_pixel_fond_noir] #élimilne le fond noir


        liste_quantile_np = np.linspace(0, 1, 1000)

        print("Calcul des quantiles Source...")
        value_quantile_source = torch.from_numpy(
            np.quantile(source_pixel_breast, liste_quantile_np)
        ).float()

        print("Calcul des quantiles Target...")
        value_quantile_target = torch.from_numpy(
            np.quantile(target_pixel_breast, liste_quantile_np)
        ).float()
        
        return(value_quantile_source,value_quantile_target)
    
    (value_quantile_source,value_quantile_target)=calcul_quantile_et_representation(representation_graphique=True)

    #--------------------------------------------------------------------------------------------------------------------------------
    def apply_ot_mapping(image_tensor):
        image_flatten=image_tensor.flatten()
        img_flat_np = image_flatten.numpy()
        mask = (img_flat_np > seuil_pixel_fond_noir) & (img_flat_np < seuil_brillance)
        img_flat_np[mask] = np.interp(
            img_flat_np[mask], 
            value_quantile_target.numpy(), 
            value_quantile_source.numpy()
        )
        return(torch.from_numpy(img_flat_np).view_as(image_tensor))
    #-------------------------------------------------------------------------------------------------------------------------------
    #application des fonctions annexes pour générer les données transformées
    if choix_normalisation_image==0:
        def identity(x):
            return x
        fonction = identity
    elif choix_normalisation_image==1:
        fonction=robust_standardize1
    else:
        (mu,sigma)=calcul_dataset_mean_std()
        fonction=robust_standardize2
    

    new_transform2 = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor(),
        transforms.Lambda(fonction),
        transforms.Lambda(lambda x : apply_ot_mapping(x)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    data_set=ImageFolder(root=chemin_data_vindr, transform=new_transform2)
    label=np.array(data_set.targets)
    print(len(label))
    print(label.sum()/len(label))

    test_loader_vindr_matchin_quantile = DataLoader(
        data_set, 
        batch_size=bs,       # Augmente le batch_size si possible (ex: 64 ou 128)
        shuffle=False, 
        num_workers=4,       # <--- ESSENTIEL : utilise plusieurs cœurs CPU
        pin_memory=True      # <--- Accélère le transfert CPU vers GPU
    )

    return(test_loader_vindr_matchin_quantile)

In [ ]:
test_loader_vindr_matchin_quantile=generation_donnes_eval_matching_quantile(seuil_pixel_fond_noir=-2.8,seuil_brillance=2.9)
evaluation(choose_data=test_loader_vindr_matchin_quantile)

In [ ]:
#visualisation images avant après matching quantile

images, labels = next(iter(test_loader_vindr_matchin_quantile))
img = images[10] 


# 3. Dé-normalisation pour l'affichage (optionnel mais recommandé)
# Si on veut juste voir l'effet du mapping, on peut simplement rescale entre 0 et 1
img_to_show = (img - img.min()) / (img.max() - img.min())


# 4. Affichage
# Attention : PyTorch utilise (C, H, W), Matplotlib veut (H, W, C)
plt.figure(figsize=(4, 4))
plt.imshow(img_to_show.permute(1, 2, 0).cpu().numpy())
plt.title(f"Image VinDr avec OT Mapping - Label: {labels[0]}")
plt.axis('off')
plt.show()



images, labels = next(iter(test_loader_vindr))
img = images[10] 

# 3. Dé-normalisation pour l'affichage (optionnel mais recommandé)
# Si on veut juste voir l'effet du mapping, on peut simplement rescale entre 0 et 1
img_to_show = (img - img.min()) / (img.max() - img.min())

# 4. Affichage
# Attention : PyTorch utilise (C, H, W), Matplotlib veut (H, W, C)
plt.figure(figsize=(4, 4))
plt.imshow(img_to_show.permute(1, 2, 0).cpu().numpy())
plt.title(f"Image VinDr sans OT Mapping - Label: {labels[0]}")
plt.axis('off')
plt.show()


test_loader_miniddsm0=generation_donnes_eval(chemin_data=chemin_data_miniddsm,mode=0)
images, labels = next(iter(test_loader_miniddsm0))
img = images[3] 

# 3. Dé-normalisation pour l'affichage (optionnel mais recommandé)
# Si on veut juste voir l'effet du mapping, on peut simplement rescale entre 0 et 1
img_to_show = (img - img.min()) / (img.max() - img.min())

# 4. Affichage
# Attention : PyTorch utilise (C, H, W), Matplotlib veut (H, W, C)
plt.figure(figsize=(4, 4))
plt.imshow(img_to_show.permute(1, 2, 0).cpu().numpy())
plt.title(f"Image miniDDSM - Label: {labels[0]}")
plt.axis('off')
plt.show()